# Lab 8: PostgreSQL and Spark SQL on the same flights

Run the cells one at a time. The notebook runs the same two queries in PostgreSQL and in Spark SQL on 1, 5, 10, 15 and 20 million flights, and ends with a table that compares the times. The web UI of Spark is at <http://localhost:4040> while the session is open.

## Connect to both engines

PostgreSQL is the local service of the development container. Spark runs on all the cores of the container.

In [ ]:
import glob
import shutil
import statistics
import subprocess
import time

import pandas as pd
import psycopg
from pyspark.sql import SparkSession

pg = psycopg.connect(autocommit=True)
spark = (SparkSession.builder.master('local[*]').appName('lab08').config(
    'spark.driver.memory', '4g').config('spark.sql.adaptive.enabled',
                                        'false').getOrCreate())
print(pg.info.parameter_status('server_version'), spark.version,
      spark.sparkContext.defaultParallelism)

## The two queries

The analytic query counts, for each airline, its flights and the ones that left more than 15 minutes late. The lookup finds one flight by its `flight_id`. Both engines run exactly the same text.

In [ ]:
ANALYTIC = '''
SELECT airline_code,
       count(*) AS flights,
       sum(CASE WHEN actual_departure > scheduled_departure + INTERVAL '15' MINUTE
                THEN 1 ELSE 0 END) AS late_flights
FROM flights
WHERE cancelled = 0
GROUP BY airline_code
ORDER BY airline_code
'''


def lookup_sql(flights):
    flight_id = 1000000 + flights * 3 // 5
    return f'SELECT * FROM flights WHERE flight_id = {flight_id}'


print(ANALYTIC)
print(lookup_sql(20000000))

## Run a query and time it

`run_pg` and `run_spark` send a query to each engine and bring the rows back to Python. `seconds` runs a query three times and keeps the median.

In [ ]:
def run_pg(sql):
    with pg.cursor() as cur:
        cur.execute(sql)
        return cur.fetchall()


def run_spark(sql):
    return spark.sql(sql).collect()


def seconds(run, sql, repeats=3):
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        run(sql)
        times.append(time.perf_counter() - start)
    return statistics.median(times)

## Load the flights into both engines

For each volume, the generator writes the CSV files. PostgreSQL loads them into the table `flights` with `COPY`. Spark reads them, writes Parquet and registers the Parquet files as the view `flights`. Both loads are timed.

In [ ]:
OUT = '../shared/output/lab08'
SCHEMA = ('flight_id long, flight_date date, airline_code string, '
          'flight_number string, tail_number string, origin string, '
          'destination string, scheduled_departure timestamp, '
          'actual_departure timestamp, distance_km int, cancelled int')
CREATE_TABLE = '''
CREATE TABLE flights (
    flight_id bigint, flight_date date, airline_code text,
    flight_number text, tail_number text, origin text, destination text,
    scheduled_departure timestamp, actual_departure timestamp,
    distance_km integer, cancelled integer)
'''


def load_volume(flights):
    shutil.rmtree(OUT, ignore_errors=True)
    subprocess.run([
        'python3', '../shared/generator/generate_flights.py', '--flights',
        str(flights), '--output', OUT
    ],
                   check=True,
                   stdout=subprocess.DEVNULL)
    start = time.perf_counter()
    with pg.cursor() as cur:
        cur.execute('DROP TABLE IF EXISTS flights')
        cur.execute(CREATE_TABLE)
        for path in sorted(glob.glob(f'{OUT}/flights/*.csv')):
            with open(path, 'rb') as csv_file:
                with cur.copy('COPY flights FROM STDIN '
                              'WITH (FORMAT csv, HEADER true)') as copy:
                    while chunk := csv_file.read(1 << 20):
                        copy.write(chunk)
        cur.execute('ANALYZE flights')
    pg_load = time.perf_counter() - start
    start = time.perf_counter()
    (spark.read.csv(
        f'{OUT}/flights', header=True,
        schema=SCHEMA).write.mode('overwrite').parquet(f'{OUT}/parquet'))
    spark_load = time.perf_counter() - start
    spark.read.parquet(f'{OUT}/parquet').createOrReplaceTempView('flights')
    shutil.rmtree(f'{OUT}/flights')
    return {
        'pg_load_s': pg_load,
        'spark_parquet_s': spark_load
    }

## Measure one volume

The function loads the flights and runs the analytic query in both engines, after checking that they return the same rows. Then it runs the lookup in PostgreSQL without an index, builds the index on `flight_id`, runs the lookup again, and runs it in Spark.

In [ ]:
CREATE_INDEX = 'CREATE INDEX flights_flight_id_idx ON flights (flight_id)'


def measure(flights):
    result = {
        'flights': flights,
        **load_volume(flights)
    }
    lookup = lookup_sql(flights)
    assert run_pg(ANALYTIC) == run_spark(ANALYTIC)
    result['pg_analytic_s'] = seconds(run_pg, ANALYTIC)
    result['spark_analytic_s'] = seconds(run_spark, ANALYTIC)
    result['pg_lookup_no_index_s'] = seconds(run_pg, lookup)
    start = time.perf_counter()
    pg.execute(CREATE_INDEX)
    result['pg_index_build_s'] = time.perf_counter() - start
    result['pg_lookup_index_s'] = seconds(run_pg, lookup)
    result['spark_lookup_s'] = seconds(run_spark, lookup)
    return result

## Run the five volumes

This cell takes a few minutes. PostgreSQL keeps only the last table, the one with 20 million flights.

In [ ]:
rows = []
for flights in [1000000, 5000000, 10000000, 15000000, 20000000]:
    if rows:
        pg.execute('DROP INDEX IF EXISTS flights_flight_id_idx')
    rows.append(measure(flights))
    print(f'{flights} flights done')

## Compare the analytic query

Each ratio is the time of PostgreSQL divided by the time of Spark SQL. A ratio above 1 means that Spark SQL answered faster, and below 1 means that PostgreSQL did.

In [ ]:
results = pd.DataFrame(rows).set_index('flights')
analytic = pd.DataFrame({
    'PostgreSQL (s)': results['pg_analytic_s'],
    'Spark SQL (s)': results['spark_analytic_s']
})
analytic['PostgreSQL / Spark SQL'] = (analytic['PostgreSQL (s)'] /
                                      analytic['Spark SQL (s)'])
analytic.round(2)

## Compare the lookup

The first ratio shows how much faster PostgreSQL gets with the index. The second one divides the time of Spark SQL by the time of PostgreSQL with the index.

In [ ]:
lookup = pd.DataFrame({
    'PostgreSQL without index (ms)': results['pg_lookup_no_index_s'] * 1000,
    'PostgreSQL with index (ms)': results['pg_lookup_index_s'] * 1000,
    'Spark SQL (ms)': results['spark_lookup_s'] * 1000
})
lookup['without index / with index'] = (
    lookup['PostgreSQL without index (ms)'] /
    lookup['PostgreSQL with index (ms)'])
lookup['Spark SQL / with index'] = (lookup['Spark SQL (ms)'] /
                                    lookup['PostgreSQL with index (ms)'])
lookup.round(2)

## Compare the loads

PostgreSQL copies the CSV files into a table. Spark converts them to Parquet. The index is the cost PostgreSQL pays to make the lookup fast.

In [ ]:
loads = pd.DataFrame({
    'PostgreSQL COPY (s)': results['pg_load_s'],
    'PostgreSQL index (s)': results['pg_index_build_s'],
    'Spark Parquet (s)': results['spark_parquet_s']
})
loads.round(2)

## The plans

The plans of the last volume. PostgreSQL shows its plan with `EXPLAIN ANALYZE`, and Spark SQL with `explain()`.

In [ ]:
def explain_pg(sql):
    for (line, ) in run_pg(f'EXPLAIN ANALYZE {sql}'):
        print(line)


explain_pg(ANALYTIC)

In [ ]:
spark.sql(ANALYTIC).explain()

The lookup in PostgreSQL, first with the index and then without it.

In [ ]:
lookup_query = lookup_sql(20000000)
explain_pg(lookup_query)
pg.execute('DROP INDEX flights_flight_id_idx')
explain_pg(lookup_query)

In [ ]:
spark.sql(lookup_query).explain()

## Stop Spark and remove the table

In [ ]:
pg.execute('DROP TABLE IF EXISTS flights')
pg.close()
spark.stop()
shutil.rmtree(OUT, ignore_errors=True)
spark